<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/develop/llm_otus_filippov_hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Шаг 1: Установка необходимых библиотек

    Нам потребуются следующие пакеты:
    ┌───────────────────────┬───────────────────────────────────────────────────────┐
    │ Библиотека            │ Зачем                                                 │
    ├───────────────────────┼───────────────────────────────────────────────────────┤
    │ datasets              │ Загрузка датасета Sberquad из HuggingFace             │
    │ transformers + torch  │ Запуск локальных языковых моделей                     │
    │ evaluate, nltk        │ Расчёт BLEU-метрики                                   │
    │ sentence-transformers │ Semantic Similarity — косинусная близость эмбеддингов │
    │ pandas                │ Хранение и агрегация результатов                      │
    │ matplotlib, seaborn   │ Визуализация и сравнение моделей                      │

In [1]:
!pip install datasets transformers torch evaluate nltk sacrebleu sentence-transformers tqdm pandas matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00


Импорт зависимостей

    Импортируем всё необходимое для дальнейшей работы.
    
    ------
  
    Настраиваем seaborn для красивых график и скачиваем punkt для токенизации NLTK.

In [4]:
import os
import time
import re
import string
from collections import Counter

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from datasets import load_dataset

from transformers import (
  AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer, util

import nltk
nltk.download('punkt', quiet=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

Загружаем датасет kuznetsoffandrey/sberquad — это русскоязычный QA-датасет, аналог SQuAD. Каждый пример содержит:

     - context — текст-контекст
     - question — вопрос по контексту
     - answers — эталонные ответы (список с позициями)

In [5]:
dataset = load_dataset("kuznetsoffandrey/sberquad")
print("Доступные сплиты:", dataset.keys())
print("\nСтруктура:")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sberquad/train-00000-of-00001.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

sberquad/validation-00000-of-00001.parqu(…):   0%|          | 0.00/3.43M [00:00<?, ?B/s]

sberquad/test-00000-of-00001.parquet:   0%|          | 0.00/4.93M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45328 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5036 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/23936 [00:00<?, ? examples/s]

Доступные сплиты: dict_keys(['train', 'validation', 'test'])

Структура:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 45328
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5036
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 23936
    })
})


Sberquad имеет вложенную структуру — answers это словарь с ключами text (список ответов) и answer_start (позиции). Убедимся, что правильно понимаем формат.

In [6]:
sample = dataset['validation'][0]
print("Контекст:", sample['context'][:200], "...")
print("\nВопрос:", sample['question'])
print("\nОтветы:", sample['answers'])

Контекст: Первые упоминания о строении человеческого тела встречаются в Древнем Египте. В XXVII веке до н. э. египетский врач Имхотеп описал некоторые органы и их функции, в частности головной мозг, деятельност ...

Вопрос: Где встречаются первые упоминания о строении человеческого тела?

Ответы: {'text': ['в Древнем Египте'], 'answer_start': [60]}


Подготовка выборки из 200 примеров

    Формируем DataFrame для оценки. Берём сплит validation — он меньше и репрезентативнее для тестирования. В качестве эталонного ответа используем первый вариант из списка ответов.

In [10]:
NUM_SAMPLES = 50

data = dataset['validation']
data_subset = data.select(range(min(NUM_SAMPLES, len(data))))

samples = []
for item in data_subset:
  gt_answers = item['answers']['text']
  if gt_answers and len(gt_answers) > 0:
     samples.append({
     'context': item['context'],
     'question': item['question'],
     'ground_truth': gt_answers[0]
     })

df = pd.DataFrame(samples)
print(f"Всего примеров для оценки: {len(df)}")
df.head()


Всего примеров для оценки: 50


,context,question,ground_truth
0,Первые упоминания о строении человеческого тел...,Где встречаются первые упоминания о строении ч...,в Древнем Египте
1,Первые упоминания о строении человеческого тел...,Когда египетский врач Имхотеп впервые описал н...,В XXVII веке до н. э.
2,Телескоп имеет модульную структуру и содержит ...,Как называется корректирующая оптическая систе...,COSTAR
3,Критики теории Вегенера поставили во главу угл...,Какая теория была отвергнута после смерти Веге...,теория дрейфа материков
4,При нагревании кусочки янтаря становятся очень...,Чему не уступают по красоте изделия из прессов...,изделиям из монолитных камней


Шаг 3: Определение устройства вычислений

    Проверяем, доступен ли GPU. Все модели будут загружены на доступное устройство. Если только CPU — генерация 200 примеров тремя моделями будет очень медленной

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

if device.type == 'cpu':
   print("⚠️ GPU не обнаружен. Генерация на CPU может быть очень медленной.")
   print("   Рекомендуется Google Colab с GPU или уменьшение NUM_SAMPLES до 20-50")

Используемое устройство: cpu
⚠️ GPU не обнаружен. Генерация на CPU может быть очень медленной.
   Рекомендуется Google Colab с GPU или уменьшение NUM_SAMPLES до 20-50


Авторизуемся на hugging face

In [16]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)


Модель 1: ruGPT-3 Medium

Русскоязычная GPT среднего размера (~400M). Легче и быстрее, чем Large версия. Causal LM — генерирует продолжение текста.

In [21]:
print("Загрузка ruGPT3-Medium...")

rugpt_tokenizer = AutoTokenizer.from_pretrained('ai-forever/rugpt3medium_based_on_gpt2')
rugpt_model = AutoModelForCausalLM.from_pretrained(
    'ai-forever/rugpt3medium_based_on_gpt2',
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map='auto' if device.type == 'cuda' else None
    )

if device.type == 'cpu':
   rugpt_model = rugpt_model.to(device)

rugpt_model.eval()
print("✅ ruGPT3-Medium загружена")

Загрузка ruGPT3-Medium...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3medium_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ruGPT3-Medium загружена


Модель 2: ruGPT-3 Large

Большая русскоязычная GPT (~760M). Более качественная генерация, но медленнее. Тоже causal LM.

In [22]:
print("Загрузка ruGPT3-Large...")

rugpt2_tokenizer = AutoTokenizer.from_pretrained('ai-forever/rugpt3large_based_on_gpt2')
rugpt2_model = AutoModelForCausalLM.from_pretrained(
     'ai-forever/rugpt3large_based_on_gpt2',
      torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
      device_map='auto' if device.type == 'cuda' else None
      )

if device.type == 'cpu':
  rugpt2_model = rugpt2_model.to(device)

rugpt2_model.eval()
print("✅ ruGPT3-Large загружена")

Загрузка ruGPT3-Large...


config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ruGPT3-Large загружена
